# Bond-order mismatch viewer (rdmol_smiles vs rdmol_xyz)

This notebook loads failing entries from `output_xyz.json` and `output_no_excuse.json` and groups them into:
- `xyz_more_than_ref`
- `xyz_equal_ref`
- `xyz_less_than_ref`

Then you can flip through entries and visually compare `rdmol_smiles` (reference) vs `rdmol_xyz` (predicted).


In [1]:
import json
from pathlib import Path
from collections import Counter

import h5py
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
from IPython.display import display, Markdown
import ipywidgets as widgets
import networkx as nx

import tmos

from openff.toolkit import Molecule

NOTEBOOK_DIR = Path.cwd()
SPICE_PATH = "/Users/jenniferclark/bin/back-to-school-jen/1_data/SPICE-2.0.1.hdf5"

def charged_element_signature(mol):
    """Count charged atoms by element and formal charge.

    Returns Counter[(atomic_num, formal_charge)] for non-neutral atoms.
    """
    return Counter(
        (atom.GetAtomicNum(), atom.GetFormalCharge())
        for atom in mol.GetAtoms()
        if atom.GetFormalCharge() != 0
    )


In [2]:
_xyz_records = json.loads((NOTEBOOK_DIR / "output_xyz.json").read_text())
_no_excuse_records = json.loads((NOTEBOOK_DIR / "output_no_excuse.json").read_text())

# Build a key→record mapping (deduplicated; no_excuse takes precedence)
json_records = {r["key"]: r for r in [*_xyz_records, *_no_excuse_records]}
keys = list(json_records.keys())
print(f"Loaded {len(keys)} records: {len(_xyz_records)} from output_xyz.json, {len(_no_excuse_records)} from output_no_excuse.json")


Loaded 101 records: 100 from output_xyz.json, 1 from output_no_excuse.json


In [3]:
def check_nx_iso(mol1, mol2):
    nx_smiles = tmos.graph_mapping.mol_to_graph(mol1)
    nx_xyz    = tmos.graph_mapping.mol_to_graph(mol2)
    node_match = nx.algorithms.isomorphism.categorical_node_match("symbol", None)
    gm = nx.algorithms.isomorphism.GraphMatcher(nx_smiles, nx_xyz, node_match=node_match)
    return gm.is_isomorphic()
    
def check_offmol_iso(mol1, mol2):
    flag, _ = Molecule.are_isomorphic(
        Molecule.from_rdkit(mol1, allow_undefined_stereo=True, hydrogens_are_explicit=True), 
        Molecule.from_rdkit(mol2, allow_undefined_stereo=True, hydrogens_are_explicit=True),
        return_atom_map=True,
        aromatic_matching=True,
        atom_stereochemistry_matching=False,
        bond_stereochemistry_matching=False,
    )
    return flag


def _n_charged(mol):
    return sum(1 for atom in mol.GetAtoms() if atom.GetFormalCharge() != 0)


def check_new_xyz_to_mol(record):
    offmol_cmiles = Molecule.from_mapped_smiles(record["smiles"], allow_undefined_stereo=True)
    rdmol_cmiles = offmol_cmiles.to_rdkit()
    Chem.RemoveStereochemistry(rdmol_cmiles)

    rdmol_tmos = tmos.build_rdmol.xyz_to_rdkit(
        record["symbols"], record["coordinates"], ignore_scale=True
    )
    rdmol_tmos = tmos.build_rdmol.determine_bonds(rdmol_tmos, charge=record["charge"])
    Chem.RemoveStereochemistry(rdmol_tmos)

    return {
        "smiles": offmol_cmiles,
        "rdmol_smiles": rdmol_cmiles,
        "rdmol_xyz": rdmol_tmos,
        "iso_nx": check_nx_iso(rdmol_cmiles, rdmol_tmos),
        "iso_off": check_offmol_iso(rdmol_cmiles, rdmol_tmos),
    }


def compute_mismatch_buckets():
    buckets = {
        "xyz_more_than_ref": [],
        "xyz_equal_ref": [],
        "xyz_less_than_ref": [],
        "resonance_equivalent": [],
        "smiles_is_worse": [],
    }

    with h5py.File(SPICE_PATH) as spice:
        for key, record in json_records.items():
            smiles = spice[key]["smiles"].asstr()[0]
            data = {**record, "smiles": smiles}
            output = check_new_xyz_to_mol(data)

            if not output["iso_nx"]:
                continue
            if output["iso_off"]:
                continue

            ref = output["rdmol_smiles"]
            xyz = output["rdmol_xyz"]
            ref_chg = _n_charged(ref)
            xyz_chg = _n_charged(xyz)

            item = {
                "key": key,
                "ref": ref,
                "xyz": xyz,
                "ref_charged": ref_chg,
                "xyz_charged": xyz_chg,
                "ref_smiles": Chem.MolToSmiles(ref, canonical=True),
                "xyz_smiles": Chem.MolToSmiles(xyz, canonical=True),
            }

            sig_ref = charged_element_signature(output["rdmol_smiles"])
            sig_xyz = charged_element_signature(output["rdmol_xyz"])
            penalty_ref = tmos.build_rdmol.molecular_penalty(output["rdmol_smiles"])
            penalty_xyz = tmos.build_rdmol.molecular_penalty(output["rdmol_xyz"])

            if sig_ref == sig_xyz:
                buckets["resonance_equivalent"].append(item)
            if penalty_ref > penalty_xyz:
                buckets["smiles_is_worse"].append(item)
            elif xyz_chg > ref_chg:
                buckets["xyz_more_than_ref"].append(item)
            elif xyz_chg == ref_chg:
                buckets["xyz_equal_ref"].append(item)
            else:
                buckets["xyz_less_than_ref"].append(item)

    return buckets


buckets = compute_mismatch_buckets()
counts = {k: len(v) for k, v in buckets.items()}
counts, sum(counts.values())


/Users/jenniferclark/mamba/envs/architector/lib/python3.11/site-packages/openff/toolkit/topology/molecule.py:4341: MultipleComponentsInMoleculeWarning: RDKit Molecule passed to from_rdkit consists of more than one molecule, consider running rdkit.Chem.AllChem.GetMolFrags(rdmol, asMols=True) or splitting input SMILES at '.' to get separate molecules and pass them to from_rdkit one at a time. While this is supported for legacy reasons, OpenFF Molecule objects are not supposed to contain disconnected chemical graphs and this may result in undefined behavior later on. The OpenFF ecosystem is built to handle multiple molecules, but they should be in a Topology object, ex: top = Topology.from_molecules([mol1, mol2])
  molecule = toolkit.from_rdkit(
/Users/jenniferclark/mamba/envs/architector/lib/python3.11/site-packages/openff/toolkit/topology/molecule.py:4341: MultipleComponentsInMoleculeWarning: RDKit Molecule passed to from_rdkit consists of more than one molecule, consider running rdkit.

({'xyz_more_than_ref': 0,
  'xyz_equal_ref': 1,
  'xyz_less_than_ref': 0,
  'resonance_equivalent': 1,
  'smiles_is_worse': 0},
 2)

In [4]:
display(Markdown(
    f"**Mismatch counts**\n\n"
    f"- resonance_equivalent: {len(buckets['resonance_equivalent'])}\n"
    f"- smiles_is_worse: {len(buckets['smiles_is_worse'])}\n"
    f"- xyz_more_than_ref: {len(buckets['xyz_more_than_ref'])}\n"
    f"- xyz_equal_ref: {len(buckets['xyz_equal_ref'])}\n"
    f"- xyz_less_than_ref: {len(buckets['xyz_less_than_ref'])}\n"
    f"- total: {sum(len(v) for v in buckets.values())}"
))

category = widgets.Dropdown(
    options=["xyz_more_than_ref", "xyz_equal_ref", "xyz_less_than_ref", "smiles_is_worse", "resonance_equivalent"],
    value="xyz_more_than_ref",
    description="Category:",
    layout=widgets.Layout(width="350px"),
)

entry_idx = widgets.IntSlider(
    value=0, min=0, max=max(0, len(buckets[category.value]) - 1), step=1,
    description="Entry:",
    continuous_update=False,
    layout=widgets.Layout(width="500px"),
)

prev_btn = widgets.Button(description="◀ Prev", button_style="")
next_btn = widgets.Button(description="Next ▶", button_style="")

out = widgets.Output()


def _render():
    with out:
        out.clear_output(wait=True)
        items = buckets[category.value]
        n = len(items)
        if n == 0:
            print("No entries in this category.")
            return

        idx = min(entry_idx.value, n - 1)
        item = items[idx]

        print(f"Category: {category.value} | Entry: {idx + 1}/{n} | Key: {item['key']}")
        print(f"Charged atoms  ref={item['ref_charged']}  xyz={item['xyz_charged']}")
        print("ref_smiles:", item['ref_smiles'])
        print("xyz_smiles:", item['xyz_smiles'])

        # prepare flattened 2D depictions by computing 2D coords
        ref2d = Chem.Mol(item['ref'])
        AllChem.Compute2DCoords(ref2d)
        xyz2d = Chem.Mol(item['xyz'])
        AllChem.Compute2DCoords(xyz2d)

        img = Draw.MolsToGridImage(
            [ref2d, xyz2d],
            molsPerRow=2,
            subImgSize=(500, 380),
            legends=["rdmol_smiles (ref)", "rdmol_xyz"],
            useSVG=True,
        )
        display(img)


def _on_category_change(change):
    if change['name'] == 'value':
        n = len(buckets[change['new']])
        entry_idx.max = max(0, n - 1)
        entry_idx.value = 0
        _render()


def _on_idx_change(change):
    if change['name'] == 'value':
        _render()


def _on_prev(_):
    if entry_idx.value > 0:
        entry_idx.value -= 1


def _on_next(_):
    if entry_idx.value < entry_idx.max:
        entry_idx.value += 1


category.observe(_on_category_change, names='value')
entry_idx.observe(_on_idx_change, names='value')
prev_btn.on_click(_on_prev)
next_btn.on_click(_on_next)

controls = widgets.VBox([
    widgets.HBox([category]),
    widgets.HBox([entry_idx, prev_btn, next_btn]),
])
display(out, controls)
_render()
# render initial entry

**Mismatch counts**

- resonance_equivalent: 1
- smiles_is_worse: 0
- xyz_more_than_ref: 0
- xyz_equal_ref: 1
- xyz_less_than_ref: 0
- total: 2

Output()